# Tutorial 8: Three-Model Coupling

Estimated time: 25-45 minutes

## Prerequisites
No extra dependencies beyond base package runtime.

## Learning aims
- Primary package aim: extend coupling graph to a third model and sample joint outputs
- Secondary scientific aim: observe uncertainty propagation through chained couplings

## Success criteria
- you can inspect downstream uncertainty in `w` and relate it to upstream coupling structure


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Build and sample 3-model metamodel


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Locates the repo root, puts src/ on sys.path, and defines helpers that run
# the `mm` CLI in-process (run_mm_cli) and auxiliary tools like pytest/ruff
# via the active interpreter (run_tool). No shell cells, no PYTHONPATH prefix.
import io
import os
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Could not locate project root (expected src/).")

src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Run the whole notebook from the repo root so CLI artifacts and any
# CWD-relative registry lookups (e.g. eval_surrogate) resolve consistently.
os.chdir(root)

# Jupyter caches imported modules; clear bayesian_metamodeling so re-runs pick
# up the current local source.
for module_name in list(sys.modules):
    if module_name == "bayesian_metamodeling" or module_name.startswith("bayesian_metamodeling."):
        del sys.modules[module_name]

from bayesian_metamodeling.cli.main import main as mm_main


@contextmanager
def in_project_root():
    previous = Path.cwd()
    os.chdir(root)
    try:
        yield
    finally:
        os.chdir(previous)


def run_mm_cli(*args: str, check: bool = True) -> int:
    """Run `mm <args>` in-process; cross-platform, no shell."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    previous_argv = sys.argv[:]
    try:
        sys.argv = ["mm", *args]
        with in_project_root(), redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exit_code = mm_main()
    finally:
        sys.argv = previous_argv

    print("$ mm", " ".join(args))
    out = stdout_buf.getvalue().strip()
    err = stderr_buf.getvalue().strip()
    if out:
        print(out)
    if err:
        print(err)
    if check and exit_code != 0:
        raise RuntimeError(f"CLI command failed ({exit_code}): mm {' '.join(args)}")
    return exit_code


def run_tool(*args: str, check: bool = True) -> int:
    """Run an auxiliary tool (pytest, ruff) via the active interpreter, cross-platform."""
    cmd = list(args)
    if cmd and cmd[0] in {"pytest", "ruff"}:
        cmd = [sys.executable, "-m", *cmd]
    print("$", " ".join(args))
    with in_project_root():
        result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Tool failed ({result.returncode}): {' '.join(args)}")
    return result.returncode


In [ ]:
run_mm_cli('meta', 'build', 'tutorials/specs/metamodel.three_model.pymc.json')
run_mm_cli('meta', 'sample', 'tutorials/specs/metamodel.three_model.pymc.json', '--draws', '200', '--tune', '100', '--chains', '2', '--seed', '321')


## Step 2: Compare uncertainty spread (graphic)


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

reg = root / 'tmp/metamodel_samples_registry.json'
payload = json.loads(reg.read_text())
latest_id = sorted(payload.keys())[-1]
dataset_path = Path(payload[latest_id]['samples_dataset_path'])
ds = json.loads(dataset_path.read_text())

vars_ = ds['variables']
for name in ['y', 'z', 'w']:
    if name not in vars_:
        print(f'Missing variable {name} in latest sample dataset')

plt.figure(figsize=(7, 4))
for name, color in [('y', 'tab:blue'), ('z', 'tab:green'), ('w', 'tab:red')]:
    if name in vars_:
        arr = np.asarray(vars_[name], dtype=float).reshape(-1)
        plt.hist(arr, bins=30, alpha=0.4, label=name, color=color)
plt.title('Distribution comparison across coupling chain')
plt.xlabel('value')
plt.ylabel('frequency')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

## Scientific checkpoint
Is downstream `w` broader/narrower than upstream variables? Explain in terms of coupling and noise.
